# 🦊 Entrenamiento (Fine-Tuning QLoRA) de BoniBot SBL en Google Colab (GPU T4 Gratis)

Este notebook entrena un modelo de lenguaje propio (**Qwen 2.5 1.5B Instruct**) especializado en los reglamentos, horarios, directorio y póliza de la **Corporación Colegio San Bonifacio de las Lanzas Bilingüe (SBL)**.

### Pasos para ejecutar en Google Colab ($0 Costo):
1. Ve a **Entorno de ejecución ➔ Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**.
2. Sube el archivo `data/sbl_training_dataset.jsonl` al panel de archivos de Colab (icono de carpeta a la izquierda).
3. Ejecuta las celdas en orden (`Shift + Enter`).
4. Al finalizar, el modelo cuantizado GGUF (`q4_k_m`) se subirá automáticamente a tu cuenta de **Hugging Face** listo para correr en tu Hugging Face Space gratuito.

In [ ]:
# 1. Instalación de Unsloth, TRL y librerías de Hugging Face (Optimizado para GPU T4 gratuita)
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
# 2. Cargar el modelo base Qwen 2.5 1.5B Instruct en 4-bit (ocupa solo ~1.8 GB de VRAM)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detección (Float16 para T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configurar adaptadores LoRA de alta eficiencia
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 3. Cargar y formatear el Dataset Oficial SBL (sbl_training_dataset.jsonl)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="sbl_training_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Ejemplos cargados para entrenamiento SBL: {len(dataset)}")
print("\nEjemplo #1 formateado:\n", dataset[0]["text"][:500])

In [ ]:
# 4. Entrenar el modelo (3 Épocas completas sobre la base de conocimiento SBL)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 5. Prueba rápida de inferencia en vivo después del entrenamiento
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Eres BoniBot 🦊, el asistente virtual oficial de la Corporación Colegio San Bonifacio de las Lanzas Bilingüe (SBL) en Ibagué, Tolima."},
    {"role": "user", "content": "¿A qué hora atiende a padres el profesor Juan David de TechnoScience?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.2)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 6. Exportar a GGUF (Q4_K_M) y publicar en tu cuenta de Hugging Face
# Reemplaza TU_USUARIO_HF y TU_TOKEN_HF (con permisos de escritura desde https://huggingface.co/settings/tokens)
HF_USERNAME = "JuanDavidSBL" # <-- Cambia por tu usuario de Hugging Face
HF_TOKEN = "hf_..."          # <-- Pega tu token Write de Hugging Face

model.push_to_hub_gguf(
    f"{HF_USERNAME}/bonibot-sbl-1.5b-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
    token = HF_TOKEN,
)
print("✅ Modelo GGUF publicado exitosamente en Hugging Face!")